# RUNECLAW LoRA Fine-Tuning with Unsloth

Fine-tunes Llama 3.2 3B on 10,012 real trading samples from RUNECLAW.

**Runtime:** Google Colab T4 GPU (free tier works)
**Time:** ~30-45 minutes for 3 epochs
**Output:** GGUF model file → import into Ollama

## Setup

In [ ]:
# Install unsloth (optimized LoRA fine-tuning)
!pip install -q unsloth[colab-new] datasets
!pip install -q --no-deps trl peft accelerate bitsandbytes

In [ ]:
# Upload your combined_training.jsonl file
# Option 1: Use the Colab file upload button
# Option 2: Mount Google Drive

from google.colab import files
print("Upload combined_training.jsonl (11MB):")
uploaded = files.upload()
DATASET_PATH = list(uploaded.keys())[0]
print(f"Loaded: {DATASET_PATH}")

## Load Model & Apply LoRA

In [ ]:
from unsloth import FastLanguageModel
import torch

# Load base model with 4-bit quantization
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    max_seq_length=4096,
    dtype=None,  # auto-detect
    load_in_4bit=True,
)

# Apply LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                     # LoRA rank
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,           # optimized for speed
    bias="none",
    use_gradient_checkpointing="unsloth",  # 30% less VRAM
    random_state=42,
)

print(f"Trainable parameters: {model.print_trainable_parameters()}")

## Prepare Dataset

In [ ]:
from datasets import load_dataset

# Load the RUNECLAW training data
dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
print(f"Total samples: {len(dataset)}")
print(f"Columns: {dataset.column_names}")
print(f"\nSample:")
print(dataset[0])

In [ ]:
# Format into Llama 3.2 chat template
SYSTEM_PROMPT = """You are RUNECLAW, an AI trading analyst. You analyze cryptocurrency markets using the GetClaw Confluence Engine (12 weighted indicators), enforce strict risk management through 23 automated checks, and generate structured trade ideas. You never execute without human confirmation. Capital preservation above all."""

def format_prompt(example):
    """Convert alpaca format to Llama 3.2 chat format."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
    ]
    
    # Combine instruction + input as user message
    user_msg = example["instruction"]
    if example.get("input") and example["input"].strip():
        user_msg += "\n\n" + example["input"]
    
    messages.append({"role": "user", "content": user_msg})
    messages.append({"role": "assistant", "content": example["output"]})
    
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

dataset = dataset.map(format_prompt)
print(f"Formatted {len(dataset)} samples")
print(f"\nExample formatted text (first 500 chars):")
print(dataset[0]["text"][:500])

## Train

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=4096,
    dataset_num_proc=2,
    packing=True,              # pack short samples together for efficiency
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,  # effective batch size = 8
        warmup_steps=50,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir="runeclaw-checkpoints",
        save_strategy="epoch",
    ),
)

print("Starting training...")
stats = trainer.train()
print(f"\nTraining complete!")
print(f"  Loss: {stats.training_loss:.4f}")
print(f"  Runtime: {stats.metrics['train_runtime']:.0f}s")
print(f"  Samples/sec: {stats.metrics['train_samples_per_second']:.1f}")

## Test Before Export

In [ ]:
# Quick test before exporting
FastLanguageModel.for_inference(model)

test_messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": """Analyze BTC/USDT for a potential trade.

Market data:
- Price: $67,432.10
- RSI-14: 28.5 (oversold)
- MACD Histogram: 125.3 (positive)
- Bollinger %B: 0.15
- ADX: 34, +DI > -DI (TREND_UP)
- Price is at 61.8% Fibonacci retracement
- Volume spike with price increase
- EMA9 > EMA21
- OBV rising
- Price above VWAP"""}
]

inputs = tokenizer.apply_chat_template(
    test_messages, tokenize=True, add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    input_ids=inputs, max_new_tokens=1024,
    temperature=0.3, top_p=0.9,
)

response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
print("═" * 60)
print("RUNECLAW Response:")
print("═" * 60)
print(response)

## Export to GGUF (for Ollama)

In [ ]:
# Export to GGUF format — this is what Ollama uses
# Q4_K_M is a good balance of quality and size (~2GB)

print("Exporting to GGUF (Q4_K_M quantization)...")
print("This takes 5-10 minutes.\n")

model.save_pretrained_gguf(
    "runeclaw-gguf",
    tokenizer,
    quantization_method="q4_k_m",  # good quality, ~2GB
)

print("\nGGUF export complete!")
print("File: runeclaw-gguf/unsloth.Q4_K_M.gguf")

In [ ]:
# Also export Q8_0 for higher quality (larger file ~3.5GB)
model.save_pretrained_gguf(
    "runeclaw-gguf-q8",
    tokenizer,
    quantization_method="q8_0",
)
print("Q8_0 export complete!")
print("File: runeclaw-gguf-q8/unsloth.Q8_0.gguf")

In [ ]:
# Download the GGUF files
from google.colab import files

print("Downloading Q4_K_M (recommended, ~2GB)...")
files.download("runeclaw-gguf/unsloth.Q4_K_M.gguf")

# Uncomment to also download the larger Q8 version:
# print("Downloading Q8_0 (higher quality, ~3.5GB)...")
# files.download("runeclaw-gguf-q8/unsloth.Q8_0.gguf")

## Import into Ollama

After downloading the GGUF file, run these commands on your local machine:

```bash
# Create a Modelfile that points to your fine-tuned GGUF
cat > Modelfile.finetuned << 'EOF'
FROM ./unsloth.Q4_K_M.gguf

PARAMETER temperature 0.3
PARAMETER top_p 0.9
PARAMETER num_ctx 4096
PARAMETER stop "<|eot_id|>"
PARAMETER stop "<|end|>"

SYSTEM """You are RUNECLAW, an AI trading analyst. You analyze cryptocurrency markets using the GetClaw Confluence Engine (12 weighted indicators), enforce strict risk management through 23 automated checks, and generate structured trade ideas. You never execute without human confirmation. Capital preservation above all."""
EOF

# Create the model in Ollama
ollama create pbdes2022/HUMANOID-TRADERS -f Modelfile.finetuned

# Test it
ollama run pbdes2022/HUMANOID-TRADERS "Scan BTC/USDT for trade setups"

# Push to Ollama registry (optional)
ollama push pbdes2022/HUMANOID-TRADERS
```

## Optional: Push to Hugging Face Hub

In [ ]:
# Optional: Push LoRA adapter to Hugging Face
# Uncomment and set your HF token

# HF_TOKEN = "hf_your_token_here"
# model.push_to_hub_gguf(
#     "pbdes2022/HUMANOID-TRADERS-GGUF",
#     tokenizer,
#     quantization_method="q4_k_m",
#     token=HF_TOKEN,
# )
# print("Pushed to HuggingFace!")